# 🎬 AimIndia Video Studio - Watermark Remover & Logo Overlay
**Automated Headless & Interactive Video Processing powered by Kaggle GPU (T4 x2)**

### Features:
- **Watermark Removal**: Lanczos high-quality crop & digital zoom with unsharp edge recovery.
- **Outro Trimming**: Precision cut to remove unwanted ending slides (e.g., Gemini / NotebookLM outro).
- **Custom Logo Overlay**: Transparent PNG overlay with configurable size and bottom-right offset.
- **Dual Execution Mode**:
  1. `RUN_MODE = 'interactive'`: Launches an interactive Gradio UI for fast testing and preview.
  2. `RUN_MODE = 'batch'`: Headless cloud batch processor automatically triggered by the web studio.

In [ ]:
# 1. Environment & GPU Acceleration Setup
!nvidia-smi -L || echo '⚠️ No GPU detected. For fast NVENC encoding, enable GPU in Kaggle settings.'

!pip install -q gradio

# Install static ffmpeg build with NVIDIA NVENC GPU hardware support
import os, subprocess
if not os.path.exists('/usr/local/bin/ffmpeg_gpu/ffmpeg'):
    os.makedirs('/usr/local/bin/ffmpeg_gpu', exist_ok=True)
    print('⬇️ Downloading FFmpeg with NVENC hardware acceleration...')
    subprocess.run('wget -q https://github.com/BtbN/FFmpeg-Builds/releases/download/latest/ffmpeg-master-latest-linux64-gpl.tar.xz -O /tmp/ffmpeg.tar.xz', shell=True, check=True)
    subprocess.run('tar -xf /tmp/ffmpeg.tar.xz -C /tmp/', shell=True, check=True)
    subprocess.run('cp /tmp/ffmpeg-master-latest-linux64-gpl/bin/ffmpeg /tmp/ffmpeg-master-latest-linux64-gpl/bin/ffprobe /usr/local/bin/ffmpeg_gpu/', shell=True, check=True)

os.environ['PATH'] = '/usr/local/bin/ffmpeg_gpu:' + os.environ.get('PATH', '')

print('✅ Active FFmpeg:')
!which ffmpeg
!ffmpeg -version | head -n 1
!ffmpeg -hide_banner -encoders 2>/dev/null | grep nvenc || echo '  Notice: NVENC not visible, CPU libx264 will be used automatically.'
print('✅ Environment ready!')

In [ ]:
# 2. Mode Configuration Switch
# Set to 'batch' for headless web-driven runs, or 'interactive' for Gradio UI
RUN_MODE = 'interactive'
print(f'⚙️ Mode: {RUN_MODE}')

In [ ]:
# 3. Core Processing Engine
import os, sys, glob, shutil, tempfile, subprocess, json
from pathlib import Path

def get_video_dims(video_path):
    cmd = f"ffprobe -v error -select_streams v:0 -show_entries stream=width,height -of csv=p=0:s=x '{video_path}'"
    out = subprocess.check_output(cmd, shell=True).decode('utf-8').strip()
    w, h = out.split('x')
    return int(w), int(h)

def get_video_duration(video_path):
    cmd = f"ffprobe -v error -show_entries format=duration -of default=noprint_wrappers=1:nokey=1 '{video_path}'"
    return float(subprocess.check_output(cmd, shell=True).decode('utf-8').strip())

def build_filter(w, h, zoom_pct, logo_in, logo_width, x_offset, y_offset):
    crop_w = int(w * (1 - zoom_pct / 100))
    crop_h = int(h * (1 - zoom_pct / 100))
    crop_w -= crop_w % 2
    crop_h -= crop_h % 2

    if logo_in and os.path.exists(str(logo_in)):
        return (
            f"[0:v]crop={crop_w}:{crop_h}:0:0,scale={w}:{h}:flags=lanczos,"
            f"unsharp=5:5:0.6:5:5:0.0[base];"
            f"[1:v]format=rgba,scale={logo_width}:-1[logo];"
            f"[base][logo]overlay=W-w-{x_offset}:H-h-{y_offset}:format=auto"
        )
    else:
        return f"crop={crop_w}:{crop_h}:0:0,scale={w}:{h}:flags=lanczos,unsharp=5:5:0.6:5:5:0.0"

def render_video(video_in, logo_in, out_path, trim_sec=3.0, zoom_pct=6, logo_width=180, x_offset=20, y_offset=20):
    total_dur = get_video_duration(video_in)
    new_dur = max(0.1, total_dur - trim_sec)
    w, h = get_video_dims(video_in)
    has_logo = bool(logo_in and os.path.exists(str(logo_in)))

    filter_str = build_filter(w, h, zoom_pct, logo_in if has_logo else None, logo_width, x_offset, y_offset)
    filter_flag = '-filter_complex' if has_logo else '-vf'
    logo_arg = f"-i '{logo_in}' " if has_logo else ''

    cmd_gpu = (
        f"ffmpeg -i '{video_in}' {logo_arg}-t {new_dur:.3f} {filter_flag} '{filter_str}' "
        f"-c:v h264_nvenc -preset p6 -rc vbr -cq 19 -b:v 0 -maxrate 25M -bufsize 30M "
        f"-c:a copy '{out_path}' -y"
    )
    cmd_cpu = (
        f"ffmpeg -i '{video_in}' {logo_arg}-t {new_dur:.3f} {filter_flag} '{filter_str}' "
        f"-c:v libx264 -preset fast -crf 19 -c:a copy '{out_path}' -y"
    )

    print(f'⚡ Encoding: {Path(video_in).name} -> {Path(out_path).name}')
    res = subprocess.run(cmd_gpu, shell=True, capture_output=True)
    if res.returncode == 0:
        print('   ✅ Rendered via GPU NVENC acceleration!')
    else:
        print('   ⚠️ NVENC failed or unavailable, falling back to CPU...')
        subprocess.run(cmd_cpu, shell=True, check=True)
        print('   ✅ Rendered via CPU fallback!')
    return out_path

print('✅ Core processing functions loaded.')

In [ ]:
# 4. Headless Batch Execution & Interactive Launcher
if RUN_MODE == 'batch':
    print('🚀 RUN_MODE: BATCH (Automated Headless Processing)')
    
    # 1. Locate config recursively in /kaggle/input
    input_base = Path('/kaggle/input')
    cfg_files = list(input_base.rglob('config.json'))
    cfg = {}
    if cfg_files:
        with open(cfg_files[0]) as f:
            cfg = json.load(f)
        print(f'⚙️ Loaded config: {cfg}')
    else:
        print('⚙️ Using default parameters.')

    trim_sec = float(cfg.get('trim_sec', 3.0))
    zoom_pct = int(cfg.get('zoom_pct', 6))
    logo_width = int(cfg.get('logo_width', 180))
    x_offset = int(cfg.get('x_offset', 20))
    y_offset = int(cfg.get('y_offset', 20))

    # 2. Locate logo recursively
    img_exts = ('.png', '.jpg', '.jpeg', '.webp')
    logo_candidates = [p for p in input_base.rglob('*') if p.suffix.lower() in img_exts and p.is_file()]
    logo_path = logo_candidates[0] if logo_candidates else None
    print(f'🖼️ Logo: {logo_path}')

    # 3. Locate videos recursively
    vid_exts = ('.mp4', '.mov', '.mkv', '.webm', '.avi')
    videos = [p for p in input_base.rglob('*') if p.suffix.lower() in vid_exts and p.is_file() and not p.name.startswith('FINAL') and not p.name.startswith('CLEAN')]
    if not videos:
        raise RuntimeError('❌ No videos found in input dataset!')

    print(f'📹 Found {len(videos)} video(s) to process.')

    working_dir = Path('/kaggle/working')
    output_dir = working_dir / 'processed_videos'
    output_dir.mkdir(parents=True, exist_ok=True)

    processed_files = []
    for idx, vid in enumerate(videos, 1):
        out_filename = f'CLEAN_{vid.name}'
        out_file = output_dir / out_filename
        print(f'[{idx}/{len(videos)}] Processing: {vid.name}...')
        render_video(
            video_in=str(vid),
            logo_in=str(logo_path) if logo_path else None,
            out_path=str(out_file),
            trim_sec=trim_sec,
            zoom_pct=zoom_pct,
            logo_width=logo_width,
            x_offset=x_offset,
            y_offset=y_offset
        )
        processed_files.append(out_file)

    # If single video, also save directly as /kaggle/working/FINAL_VIDEO_CLEAN.mp4
    if len(processed_files) == 1:
        shutil.copy2(processed_files[0], working_dir / 'FINAL_VIDEO_CLEAN.mp4')
        print('✅ Saved single result to: /kaggle/working/FINAL_VIDEO_CLEAN.mp4')

    # Also create zip archive with all processed videos
    zip_path = working_dir / 'all_final_videos'
    shutil.make_archive(str(zip_path), 'zip', str(output_dir))
    print(f'✅ All {len(processed_files)} video(s) processed and zipped to: {zip_path}.zip')

